In [19]:
import requests
import os
import pandas as pd
import matplotlib.pyplot as plt
import random
from datetime import datetime

ROOT = ".." # Adjust to repository root

from dotenv import load_dotenv
DOTENV_PATH = os.path.join(ROOT,"../../apis/.env") # Adjust to .env file location

if load_dotenv(DOTENV_PATH):
    abs_path = os.path.abspath(DOTENV_PATH)
    drive, rel = os.path.splitdrive(abs_path)
    parts = rel.strip(os.sep).split(os.sep)
    masked = parts.copy()
    masked_parts = 1  # adjust to mask parts of your path
    max_maskable = max(0, len(parts) - 2)
    for i in range(min(masked_parts, max_maskable)):
        idx = len(parts) - 3 - i
        masked[idx] = "***"
    prefix = f"{drive}{os.sep}" if drive else (os.sep if abs_path.startswith(os.sep) else "")
    print(f"Loaded .env from {prefix}{os.sep.join(masked)}")
else:
    print("Failed to load .env file.")

API_KEY = os.getenv("AEMET_API_KEY")

def mask_token(token, unmasked_chars=3):
    return token[:unmasked_chars] + '*' * (len(token) - unmasked_chars*2) + token[-unmasked_chars:]
print(f"AEMET_API_KEY: {mask_token(API_KEY)}")

Loaded .env from c:\Users\david\***\apis\.env
AEMET_API_KEY: eyJ***********************************************************************************************************************************************************************************************************************************************************************************************************rF4


## Helper functions

In [33]:
BASE_URL = "https://opendata.aemet.es/opendata/api"
HEADERS = {"api_key": API_KEY}

def _get_data_url(endpoint: str, params: dict | None = None) -> str | None:
    resp = requests.get(f"{BASE_URL}/{endpoint}", headers=HEADERS, params=params)
    resp.raise_for_status()
    payload = resp.json()
    return payload.get("datos")

def _download_json(url: str):
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.json()

def fetch_metadata(endpoint: str) -> tuple[dict, pd.DataFrame]:
    response = requests.get(f"{BASE_URL}/{endpoint}", headers=HEADERS)
    response.raise_for_status()
    meta_url = response.json().get("metadatos")
    if not meta_url:
        return {}, pd.DataFrame()
    metadata = _download_json(meta_url)
    return metadata

## Current observations for all stations

In [31]:
datos_url = _get_data_url("observacion/convencional/todas")
observations = _download_json(datos_url)

print(len(observations))
print(observations[0])

10251
{'idema': '0009X', 'lon': 0.963335, 'fint': '2026-01-29T03:00:00+0000', 'prec': 0.0, 'alt': 406.0, 'vmax': 12.2, 'vv': 5.0, 'dv': 275.0, 'lat': 41.213892, 'dmax': 275.0, 'ubi': 'ALFORJA', 'hr': 69.0, 'tamin': 7.4, 'ta': 7.4, 'tamax': 8.3}


Field info:

In [35]:
metadata_response = fetch_metadata("observacion/convencional/todas")
print(metadata_response)

{'unidad_generadora': 'Servicio de Observación', 'periodicidad': 'continuamente', 'formato': 'application/json', 'copyright': '© AEMET. Autorizado el uso de la información y su reproducción citando a AEMET como autora de la misma.', 'notaLegal': 'https://www.aemet.es/es/nota_legal', 'campos': [{'id': 'idema', 'descripcion': 'Indicativo climatógico de la estación meteorológia automática', 'tipo_datos': 'string', 'requerido': True}, {'id': 'lon', 'descripcion': 'Longitud de la estación meteorológica (grados)', 'tipo_datos': 'float', 'requerido': True}, {'id': 'lat', 'descripcion': 'Latitud de la estación meteorológica (grados)', 'tipo_datos': 'float', 'requerido': True}, {'id': 'alt', 'descripcion': 'Altitud de la estación en metros', 'tipo_datos': 'float', 'requerido': True}, {'id': 'ubi', 'descripcion': 'Ubicación de la estación. Nombre de la estación', 'tipo_datos': 'string', 'requerido': True}, {'id': 'fint', 'descripcion': 'Fecha hora final del período de observación, se trata de da

In [44]:
fields_info = metadata_response.get("campos", [])
main_fields = ['fint', 'ubi', 'lon', 'lat', 'prec', 'ta']
candidates_name = ("nombre", "id", "campo")
candidates_desc = ("descripcion", "descripcionCampo")

for field in main_fields:
    item = next(
        (entry for entry in fields_info if any(entry.get(col) == field for col in candidates_name)),None,)
    description = (
        next((item.get(col) for col in candidates_desc if item.get(col)), None)
        if item
        else None
    )
    print(f"{field}: {description or 'Descripción no disponible'}")

fint: Fecha hora final del período de observación, se trata de datos del periodo de la hora anterior a la indicada por este campo (hora UTC)
ubi: Ubicación de la estación. Nombre de la estación
lon: Longitud de la estación meteorológica (grados)
lat: Latitud de la estación meteorológica (grados)
prec: Precipitación acumulada, medida por el pluviómetro, durante los 60 minutos anteriores a la hora indicada por el período de observación 'fint' (mm, equivalente a l/m2)
ta: Temperatura instantánea del aire correspondiente a la fecha dada por 'fint' (grados Celsius)


In [46]:
observations_df = pd.DataFrame(observations)
today_date = datetime.now().strftime("%Y%m%d")
csv_dir = os.path.join(ROOT, "csv")
os.makedirs(csv_dir, exist_ok=True)
observations_df.to_csv(os.path.join(csv_dir, f"current_observations_{today_date}.csv"), index=False)
fint_dt = pd.to_datetime(observations_df["fint"], errors="coerce")
observations_df.loc[fint_dt.notna(), "fint"] = fint_dt.dt.strftime("%Y-%m-%dT%Hh")
display(observations_df[main_fields].head(2))
display(observations_df[main_fields].tail(2))

,fint,ubi,lon,lat,prec,ta
0,2026-01-29T03h,ALFORJA,0.963335,41.213892,0.0,7.4
1,2026-01-29T03h,REUS AEROPUERTO,1.163611,41.145000,0.0,10.8


,fint,ubi,lon,lat,prec,ta
10249,2026-01-29T15h,TEGUISE LA GRACIOSA-HELIPUERTO,-13.51021,29.229585,0.0,20.4
10250,2026-01-29T15h,EL HIERRO/AEROPUERTO,-17.88889,27.818888,0.0,20.2


Daily climatological values for a station

In [31]:
def current_observations_all_stations():
    meta = requests.get(
        f"{BASE_URL}/observacion/convencional/todas",
        headers=HEADERS,
    ).json()

    datos_url = meta["datos"]
    return requests.get(datos_url).json()


data = current_observations_all_stations()

print(len(data))
print(data[0])

9398
{'idema': '0009X', 'lon': 0.963335, 'fint': '2026-01-27T22:00:00+0000', 'prec': 0.0, 'alt': 406.0, 'vmax': 14.8, 'vv': 7.4, 'dv': 266.0, 'lat': 41.213892, 'dmax': 270.0, 'ubi': 'ALFORJA', 'hr': 82.0, 'tamin': 7.1, 'ta': 7.2, 'tamax': 7.2}


Forecasts:

In [32]:
municipality_code = "28079"  # Madrid

endpoint = f"prediccion/especifica/municipio/diaria/{municipality_code}"

datos_url = _get_data_url(endpoint)
forecast = _download_json(datos_url)

print(forecast[0]["prediccion"]["dia"][0])

{'probPrecipitacion': [{'value': 0, 'periodo': '00-24'}, {'value': 0, 'periodo': '00-12'}, {'value': 95, 'periodo': '12-24'}, {'value': 0, 'periodo': '00-06'}, {'value': 100, 'periodo': '06-12'}, {'value': 0, 'periodo': '12-18'}, {'value': 90, 'periodo': '18-24'}], 'cotaNieveProv': [{'value': '', 'periodo': '00-24'}, {'value': '', 'periodo': '00-12'}, {'value': '1200', 'periodo': '12-24'}, {'value': '', 'periodo': '00-06'}, {'value': '1000', 'periodo': '06-12'}, {'value': '', 'periodo': '12-18'}, {'value': '1300', 'periodo': '18-24'}], 'estadoCielo': [{'value': '', 'periodo': '00-24', 'descripcion': ''}, {'value': '', 'periodo': '00-12', 'descripcion': ''}, {'value': '23', 'periodo': '12-24', 'descripcion': 'Intervalos nubosos con lluvia'}, {'value': '', 'periodo': '00-06', 'descripcion': ''}, {'value': '25', 'periodo': '06-12', 'descripcion': 'Muy nuboso con lluvia'}, {'value': '17', 'periodo': '12-18', 'descripcion': 'Nubes altas'}, {'value': '45n', 'periodo': '18-24', 'descripcion':